In [3]:
def compute_A1_features(row):
    # 리스트 추출
    d = row.get('A1-1_list', np.array([]))
    s = row.get('A1-2_list', np.array([]))
    r = row.get('A1-3_list', np.array([]))
    rt = row.get('A1-4_list', np.array([]))

    L = min(len(d), len(s), len(r), len(rt))
    if L == 0:
        return pd.Series({
            'A1_response_rate': 0,
            'A1_left_response_rate': 0,
            'A1_right_response_rate': 0,
            'A1_fast_response_rate': 0,
            'A1_mean_response_time': np.nan,
            'A1_fast_avg_rt': np.nan,
            'A1_direction_diff_rt': np.nan
        })

    d, s, r, rt = np.array(d[:L]), np.array(s[:L]), np.array(r[:L]), np.array(rt[:L])

    # 전체 응답률
    A1_response_rate = r.mean()

    # 왼쪽/오른쪽 조건 응답률
    A1_left_response_rate = r[d == 1].mean() if np.any(d == 1) else 0
    A1_right_response_rate = r[d == 2].mean() if np.any(d == 2) else 0

    # 빠름 조건(3)의 응답률
    A1_fast_response_rate = r[s == 3].mean() if np.any(s == 3) else 0

    # 평균 반응시간 (응답한 trial만)
    valid_rt = rt[r == 1]
    A1_mean_response_time = valid_rt.mean() if len(valid_rt) > 0 else np.nan

    # 빠름 조건에서의 평균 반응시간
    fast_rt = rt[(s == 3) & (r == 1)]
    A1_fast_avg_rt = fast_rt.mean() if len(fast_rt) > 0 else np.nan

    # 방향별 반응시간 차이 (left - right)
    left_rt = rt[(d == 1) & (r == 1)]
    right_rt = rt[(d == 2) & (r == 1)]
    A1_direction_diff_rt = left_rt.mean() - right_rt.mean() if len(left_rt) > 0 and len(right_rt) > 0 else np.nan

    return pd.Series({
        'A1_response_rate': A1_response_rate,
        'A1_left_response_rate': A1_left_response_rate,
        'A1_right_response_rate': A1_right_response_rate,
        'A1_fast_response_rate': A1_fast_response_rate,
        'A1_mean_response_time': A1_mean_response_time,
        'A1_fast_avg_rt': A1_fast_avg_rt,
        'A1_direction_diff_rt': A1_direction_diff_rt
    })

def compute_A2_features(row):
    s1 = row.get('A2-1_list', np.array([]))
    s2 = row.get('A2-2_list', np.array([]))
    r  = row.get('A2-3_list', np.array([]))
    rt = row.get('A2-4_list', np.array([]))

    L = min(len(s1), len(s2), len(r), len(rt))
    if L == 0:
        return pd.Series({
            'A2_response_rate': 0,
            'A2_slow_to_fast_rt_diff': np.nan,
            'A2_correct_ratio_by_speed': np.nan,
            'A2_mean_response_time': np.nan
        })

    s1, s2, r, rt = np.array(s1[:L]), np.array(s2[:L]), np.array(r[:L]), np.array(rt[:L])

    # 전체 응답률
    A2_response_rate = r.mean()

    # 느림/빠름 인덱스 (가정: 1=느림, 2=빠름)
    slow_idx = np.where(s1 == 1)[0]
    fast_idx = np.where(s1 == 2)[0]

    # 느림→빠름 조건 반응시간 차이
    slow_rt = rt[slow_idx & (r[slow_idx]==1)] if len(slow_idx) > 0 else np.array([])
    fast_rt = rt[fast_idx & (r[fast_idx]==1)] if len(fast_idx) > 0 else np.array([])
    A2_slow_to_fast_rt_diff = fast_rt.mean() - slow_rt.mean() if len(slow_rt) > 0 and len(fast_rt) > 0 else np.nan

    # 속도 조건별 응답률 비교 (fast/slow)
    slow_resp = r[slow_idx].mean() if len(slow_idx) > 0 else np.nan
    fast_resp = r[fast_idx].mean() if len(fast_idx) > 0 else np.nan
    if not np.isnan(slow_resp) and not np.isnan(fast_resp) and slow_resp != 0:
        A2_correct_ratio_by_speed = fast_resp / slow_resp
    else:
        A2_correct_ratio_by_speed = np.nan

    # 전체 평균 반응시간 (응답한 trial만)
    valid_rt = rt[r == 1]
    A2_mean_response_time = valid_rt.mean() if len(valid_rt) > 0 else np.nan

    return pd.Series({
        'A2_response_rate': A2_response_rate,
        'A2_slow_to_fast_rt_diff': A2_slow_to_fast_rt_diff,
        'A2_correct_ratio_by_speed': A2_correct_ratio_by_speed,
        'A2_mean_response_time': A2_mean_response_time
    })

def compute_A3_features(row):
    arrow_size     = row.get('A3-1_list', np.array([]))
    arrow_pos      = row.get('A3-2_list', np.array([]))
    arrow_dir      = row.get('A3-3_list', np.array([]))
    correct_pos    = row.get('A3-4_list', np.array([]))
    resp_type      = row.get('A3-5_list', np.array([]))
    resp           = row.get('A3-6_list', np.array([]))
    rt             = row.get('A3-7_list', np.array([]))

    L = min(len(arrow_size), len(arrow_pos), len(arrow_dir), len(correct_pos), len(resp_type), len(resp), len(rt))
    if L == 0:
        return pd.Series({
            'A3_valid_accuracy': np.nan,
            'A3_invalid_accuracy': np.nan,
            'A3_total_accuracy': np.nan,
            'A3_valid_rt': np.nan,
            'A3_invalid_rt': np.nan,
            'A3_correct_rt': np.nan,
            'A3_incorrect_rt': np.nan,
            'A3_accuracy_gap': np.nan
        })

    # 배열로 변환
    arrow_size, arrow_pos, arrow_dir = np.array(arrow_size[:L]), np.array(arrow_pos[:L]), np.array(arrow_dir[:L])
    correct_pos, resp_type, resp, rt = np.array(correct_pos[:L]), np.array(resp_type[:L]), np.array(resp[:L]), np.array(rt[:L])

    # valid / invalid trial 인덱스 (예: 1=valid, 3=invalid)
    valid_idx = np.where(resp_type == 1)[0]
    invalid_idx = np.where(resp_type == 3)[0]

    # 정확도 계산
    A3_valid_accuracy = (resp[valid_idx] == 1).mean() if len(valid_idx) > 0 else np.nan
    A3_invalid_accuracy = (resp[invalid_idx] == 1).mean() if len(invalid_idx) > 0 else np.nan
    A3_total_accuracy = (resp == 1).mean() if len(resp) > 0 else np.nan

    # 반응시간 계산 (응답한 trial만)
    A3_valid_rt = rt[valid_idx].mean() if len(valid_idx) > 0 else np.nan
    A3_invalid_rt = rt[invalid_idx].mean() if len(invalid_idx) > 0 else np.nan
    A3_correct_rt = rt[resp == 1].mean() if np.any(resp == 1) else np.nan
    A3_incorrect_rt = rt[resp == 0].mean() if np.any(resp == 0) else np.nan

    # valid / invalid 정확도 차이
    if not np.isnan(A3_valid_accuracy) and not np.isnan(A3_invalid_accuracy):
        A3_accuracy_gap = A3_valid_accuracy - A3_invalid_accuracy
    else:
        A3_accuracy_gap = np.nan

    return pd.Series({
        'A3_valid_accuracy': A3_valid_accuracy,
        'A3_invalid_accuracy': A3_invalid_accuracy,
        'A3_total_accuracy': A3_total_accuracy,
        'A3_valid_rt': A3_valid_rt,
        'A3_invalid_rt': A3_invalid_rt,
        'A3_correct_rt': A3_correct_rt,
        'A3_incorrect_rt': A3_incorrect_rt,
        'A3_accuracy_gap': A3_accuracy_gap
    })

def compute_A4_features(row):
    condition = row.get('A4-1_list', np.array([]))
    resp1     = row.get('A4-2_list', np.array([]))
    resp2     = row.get('A4-3_list', np.array([]))
    rt        = row.get('A4-4_list', np.array([]))

    L = min(len(condition), len(resp1), len(resp2), len(rt))
    if L == 0:
        return pd.Series({
            'A4_congruent_accuracy': np.nan,
            'A4_incongruent_accuracy': np.nan,
            'A4_accuracy_gap': np.nan,
            'A4_mean_rt_con': np.nan,
            'A4_mean_rt_incon': np.nan,
            'A4_rt_gap': np.nan,
            'A4_response_rate': 0
        })

    condition, resp1, resp2, rt = np.array(condition[:L]), np.array(resp1[:L]), np.array(resp2[:L]), np.array(rt[:L])

    # 응답이 있는 trial만
    valid_idx = np.where((resp1 != -1) & (resp2 != -1))[0]  # -1 등으로 결측 없음 가정
    response_rate = len(valid_idx)/L if L>0 else 0

    # congruent / incongruent trial
    con_idx = valid_idx[condition[valid_idx] == 1]
    incon_idx = valid_idx[condition[valid_idx] == 2]

    # 정확도 계산 (resp1==1이 정답)
    con_acc = (resp1[con_idx] == 1).mean() if len(con_idx) > 0 else np.nan
    incon_acc = (resp1[incon_idx] == 1).mean() if len(incon_idx) > 0 else np.nan
    acc_gap = con_acc - incon_acc if not np.isnan(con_acc) and not np.isnan(incon_acc) else np.nan

    # 반응시간 계산
    mean_rt_con = rt[con_idx].mean() if len(con_idx) > 0 else np.nan
    mean_rt_incon = rt[incon_idx].mean() if len(incon_idx) > 0 else np.nan
    rt_gap = mean_rt_incon - mean_rt_con if not np.isnan(mean_rt_con) and not np.isnan(mean_rt_incon) else np.nan

    return pd.Series({
        'A4_congruent_accuracy': con_acc,
        'A4_incongruent_accuracy': incon_acc,
        'A4_accuracy_gap': acc_gap,
        'A4_mean_rt_con': mean_rt_con,
        'A4_mean_rt_incon': mean_rt_incon,
        'A4_rt_gap': rt_gap,
        'A4_response_rate': response_rate
    })

def compute_A5_features(row):
    change_type = row.get('A5-1_list', np.array([]))
    resp1       = row.get('A5-2_list', np.array([]))
    resp2       = row.get('A5-3_list', np.array([]))
    L = min(len(change_type), len(resp1))
    if L == 0:
        return pd.Series({
            'A5_accuracy_non_change': np.nan,
            'A5_accuracy_pos_change': np.nan,
            'A5_accuracy_color_change': np.nan,
            'A5_accuracy_shape_change': np.nan,
            'A5_accuracy_var': np.nan
        })

    change_type = np.array(change_type[:L])
    resp1 = np.array(resp1[:L])

    # 각 조건별 인덱스
    idx_non_change  = np.where(change_type == 1)[0]
    idx_pos_change  = np.where(change_type == 2)[0]
    idx_color_change = np.where(change_type == 3)[0]
    idx_shape_change = np.where(change_type == 4)[0]

    # 정확도 계산 (1=정답)
    acc_non = (resp1[idx_non_change] == 1).mean() if len(idx_non_change) > 0 else np.nan
    acc_pos = (resp1[idx_pos_change] == 1).mean() if len(idx_pos_change) > 0 else np.nan
    acc_color = (resp1[idx_color_change] == 1).mean() if len(idx_color_change) > 0 else np.nan
    acc_shape = (resp1[idx_shape_change] == 1).mean() if len(idx_shape_change) > 0 else np.nan

    # 변화 유형 간 정확도 분산
    acc_list = [acc_non, acc_pos, acc_color, acc_shape]
    acc_var = np.nanvar(acc_list)  # NaN 자동 무시

    return pd.Series({
        'A5_accuracy_non_change': acc_non,
        'A5_accuracy_pos_change': acc_pos,
        'A5_accuracy_color_change': acc_color,
        'A5_accuracy_shape_change': acc_shape,
        'A5_accuracy_var': acc_var
    })

In [4]:
def compute_B1_features(row):
    r1 = row.get('B1-1_list', np.array([]))
    rt = row.get('B1-2_list', np.array([]))
    r2 = row.get('B1-3_list', np.array([]))

    L = min(len(r1), len(rt), len(r2))
    if L == 0:
        return pd.Series({
            'B1_task1_accuracy': 0,
            'B1_task2_change_acc': 0,
            'B1_task2_non_change_acc': 0,
            'B1_task2_accuracy_gap': 0,
            'B1_task2_mean_rt': np.nan
        })

    r1, r2, rt = np.array(r1[:L]), np.array(r2[:L]), np.array(rt[:L])

    # 1과제 정답률
    r1_bin = np.array([1 if val == 1 else 0 for val in r1])
    B1_task1_accuracy = r1_bin.mean()

    # 2과제: change / non-change 정확도
    r2_bin = np.array([1 if val == 1 else 0 for val in r2])
    change_mask = np.arange(L) < L//2        # 앞 절반이 change
    non_change_mask = np.arange(L) >= L//2   # 뒤 절반이 non-change

    B1_task2_change_acc = r2_bin[change_mask].mean() if np.any(change_mask) else 0
    B1_task2_non_change_acc = r2_bin[non_change_mask].mean() if np.any(non_change_mask) else 0

    # 정확도 차
    B1_task2_accuracy_gap = B1_task2_change_acc - B1_task2_non_change_acc

    # 2과제 평균 반응시간 (응답한 trial만)
    valid_rt = rt[r2_bin == 1]
    B1_task2_mean_rt = valid_rt.mean() if len(valid_rt) > 0 else np.nan

    return pd.Series({
        'B1_task1_accuracy': B1_task1_accuracy,
        'B1_task2_change_acc': B1_task2_change_acc,
        'B1_task2_non_change_acc': B1_task2_non_change_acc,
        'B1_task2_accuracy_gap': B1_task2_accuracy_gap,
        'B1_task2_mean_rt': B1_task2_mean_rt
    })

def compute_B2_features(row):
    r1 = row.get('B2-1_list', np.array([]))
    rt = row.get('B2-2_list', np.array([]))
    r2 = row.get('B2-3_list', np.array([]))

    L = min(len(r1), len(rt), len(r2))
    if L == 0:
        return pd.Series({
            'B2_task1_accuracy': 0,
            'B2_task2_change_acc': 0,
            'B2_task2_non_change_acc': 0,
            'B2_task2_accuracy_gap': 0,
            'B2_task2_mean_rt': np.nan
        })

    r1, r2, rt = np.array(r1[:L]), np.array(r2[:L]), np.array(rt[:L])

    # 1과제 정답률
    r1_bin = np.array([1 if val == 1 else 0 for val in r1])
    B2_task1_accuracy = r1_bin.mean()

    # 2과제: change / non-change 정확도
    r2_bin = np.array([1 if val == 1 else 0 for val in r2])
    change_mask = np.arange(L) < L//2
    non_change_mask = np.arange(L) >= L//2

    B2_task2_change_acc = r2_bin[change_mask].mean() if np.any(change_mask) else 0
    B2_task2_non_change_acc = r2_bin[non_change_mask].mean() if np.any(non_change_mask) else 0

    # 정확도 차
    B2_task2_accuracy_gap = B2_task2_change_acc - B2_task2_non_change_acc

    # 2과제 평균 반응시간 (응답한 trial만)
    valid_rt = rt[r2_bin == 1]
    B2_task2_mean_rt = valid_rt.mean() if len(valid_rt) > 0 else np.nan

    return pd.Series({
        'B2_task1_accuracy': B2_task1_accuracy,
        'B2_task2_change_acc': B2_task2_change_acc,
        'B2_task2_non_change_acc': B2_task2_non_change_acc,
        'B2_task2_accuracy_gap': B2_task2_accuracy_gap,
        'B2_task2_mean_rt': B2_task2_mean_rt
    })

def compute_B3_features(row):
    # 리스트 추출
    r = row.get('B3-1_list', np.array([]))
    rt = row.get('B3-2_list', np.array([]))

    L = min(len(r), len(rt))
    if L == 0:
        return pd.Series({
            'B3_accuracy': 0,
            'B3_mean_rt': np.nan
        })

    r, rt = np.array(r[:L]), np.array(rt[:L])

    # 전체 정확도
    B3_accuracy = r.mean()

    # 전체 평균 반응시간 (응답한 trial만)
    valid_rt = rt[r == 1]
    B3_mean_rt = valid_rt.mean() if len(valid_rt) > 0 else np.nan

    return pd.Series({
        'B3_accuracy': B3_accuracy,
        'B3_mean_rt': B3_mean_rt
    })

def compute_B4_features(row):
    r = row.get('B4-1_list', np.array([]))
    rt = row.get('B4-2_list', np.array([]))

    L = min(len(r), len(rt))
    if L == 0:
        return pd.Series({
            'B4_congruent_accuracy': 0,
            'B4_incongruent_accuracy': 0,
            'B4_accuracy_gap': 0,
            'B4_mean_rt_congruent': np.nan,
            'B4_mean_rt_incongruent': np.nan,
            'B4_rt_gap': np.nan
        })

    r, rt = np.array(r[:L]), np.array(rt[:L])

    # 정답 1, 오답 0 변환
    r_bin = np.array([1 if val == 1 else 0 for val in r])

    # mask 생성
    congruent_mask = np.arange(L) < L//2       # 앞 30 trials
    incongruent_mask = np.arange(L) >= L//2    # 뒤 30 trials

    # 정확도
    B4_congruent_accuracy = r_bin[congruent_mask].mean() if np.any(congruent_mask) else 0
    B4_incongruent_accuracy = r_bin[incongruent_mask].mean() if np.any(incongruent_mask) else 0
    B4_accuracy_gap = B4_incongruent_accuracy - B4_congruent_accuracy

    # 평균 반응시간 (응답한 trial만)
    valid_rt_con = rt[congruent_mask & (r_bin == 1)]
    valid_rt_incon = rt[incongruent_mask & (r_bin == 1)]

    B4_mean_rt_congruent = valid_rt_con.mean() if len(valid_rt_con) > 0 else np.nan
    B4_mean_rt_incongruent = valid_rt_incon.mean() if len(valid_rt_incon) > 0 else np.nan
    B4_rt_gap = B4_mean_rt_incongruent - B4_mean_rt_congruent if len(valid_rt_con) > 0 and len(valid_rt_incon) > 0 else np.nan

    return pd.Series({
        'B4_congruent_accuracy': B4_congruent_accuracy,
        'B4_incongruent_accuracy': B4_incongruent_accuracy,
        'B4_accuracy_gap': B4_accuracy_gap,
        'B4_mean_rt_congruent': B4_mean_rt_congruent,
        'B4_mean_rt_incongruent': B4_mean_rt_incongruent,
        'B4_rt_gap': B4_rt_gap
    })

def compute_B5_features(row):
    r = row.get('B5-1_list', np.array([]))
    rt = row.get('B5-2_list', np.array([]))

    L = min(len(r), len(rt))
    if L == 0:
        return pd.Series({
            'B5_accuracy': 0,
            'B5_mean_rt': np.nan
        })

    r, rt = np.array(r[:L]), np.array(rt[:L])

    # 정답 1, 오답 0 변환
    r_bin = np.array([1 if val == 1 else 0 for val in r])

    # 전체 정확도
    B5_accuracy = r_bin.mean()

    # 평균 반응시간 (응답한 trial만)
    valid_rt = rt[r_bin == 1]
    B5_mean_rt = valid_rt.mean() if len(valid_rt) > 0 else np.nan

    return pd.Series({
        'B5_accuracy': B5_accuracy,
        'B5_mean_rt': B5_mean_rt
    })

def compute_B6_features(row):
    r = row.get('B6_list', np.array([]))

    if len(r) == 0:
        return pd.Series({'B6_accuracy': 0})

    # 정답 1, 오답 0
    r_bin = np.array([1 if val == 1 else 0 for val in r])

    # 전체 정확도
    B6_accuracy = r_bin.mean()

    return pd.Series({'B6_accuracy': B6_accuracy})

def compute_B7_features(row):
    r = row.get('B7_list', np.array([]))

    if len(r) == 0:
        return pd.Series({'B7_accuracy': 0})

    # 정답 1, 오답 0
    r_bin = np.array([1 if val == 1 else 0 for val in r])

    # 전체 정확도
    B7_accuracy = r_bin.mean()

    return pd.Series({'B7_accuracy': B7_accuracy})

def compute_B8_features(row):
    r = row.get('B8_list', np.array([]))

    if len(r) == 0:
        return pd.Series({'B8_accuracy': 0})

    # 정답 1, 오답 0
    r_bin = np.array([1 if val == 1 else 0 for val in r])

    # 전체 정확도
    B8_accuracy = r_bin.mean()

    return pd.Series({'B8_accuracy': B8_accuracy})

In [11]:
import os, argparse, joblib
import numpy as np
import pandas as pd

from tensorflow.keras.models import load_model
from sklearn.preprocessing import StandardScaler

def safe_fromstring(x, dtype=float):
    if isinstance(x, str) and x.strip():
        return np.fromstring(x, sep=',', dtype=dtype)
    return np.array([], dtype=dtype)

def preprocess_A(train_A):
    df = train_A.copy()

    print("Step 1: Age 파생...")
    df["Age"] = df["Age"].astype(str).str.extract(r'(\d+)').astype(float)
    feats = pd.DataFrame(index=df.index)

    print("Step 2: Sequence 변환...")
    seq_cols = [
        'A1-1', 'A1-2', 'A1-3', 'A1-4',
        'A2-1', 'A2-2', 'A2-3', 'A2-4',
        'A3-1', 'A3-2', 'A3-3', 'A3-4', 'A3-5', 'A3-6', 'A3-7',
        'A4-1', 'A4-2', 'A4-3', 'A4-4', 'A4-5',
        'A5-1', 'A5-2', 'A5-3'
    ]
    for col in seq_cols:
        feats[col + '_list'] = df[col].apply(lambda x: safe_fromstring(x, dtype=float))

    print("Step 3: A1 feature 생성...")
    a1_feats = feats.apply(compute_A1_features, axis=1)
    feats = pd.concat([feats, a1_feats], axis=1)

    print("Step 4: A2 feature 생성...")
    a2_feats = feats.apply(compute_A2_features, axis=1)
    feats = pd.concat([feats, a2_feats], axis=1)

    print("Step 5: A3 feature 생성...")
    a3_feats = feats.apply(compute_A3_features, axis=1)
    feats = pd.concat([feats, a3_feats], axis=1)

    print("Step 6: A4 feature 생성...")
    a4_feats = feats.apply(compute_A4_features, axis=1)
    feats = pd.concat([feats, a4_feats], axis=1)

    print("Step 7: A5 feature 생성...")
    a5_feats = feats.apply(compute_A5_features, axis=1)
    feats = pd.concat([feats, a5_feats], axis=1)

    print("Step 8: A6 feature 생성...")
    feats['A6_score'] = df['A6-1']
    feats['A6_zscore'] = (feats['A6_score'] - feats['A6_score'].mean()) / feats['A6_score'].std()

    print("Step 9: A7 feature 생성...")
    feats['A7_score'] = df['A7-1']
    feats['A7_zscore'] = (feats['A7_score'] - feats['A7_score'].mean()) / feats['A7_score'].std()

    print("Step 10: A8 feature 생성...")
    feats['A8_distortion_score'] = df['A8-1']
    feats['A8_consistency_score'] = df['A8-2']
    feats['A8_distortion_flag'] = (feats['A8_distortion_score'] > 5).astype(int)

    print("Step 9: A7 feature 생성...")
    feats['A9_emotional_stability'] = df['A9-1']
    feats['A9_behavior_stability'] = df['A9-2']
    feats['A9_reality_checking'] = df['A9-3']
    feats['A9_cognitive_agility'] = df['A9-4']
    feats['A9_stress_level'] = df['A9-5']
    feats['A9_total_score'] = feats[['A9_emotional_stability','A9_behavior_stability',
                                        'A9_reality_checking','A9_cognitive_agility','A9_stress_level']].sum(axis=1)
    feats['A9_stability_gap'] = feats['A9_emotional_stability'] - feats['A9_behavior_stability']

    feats = feats.fillna(0)

    print("A 검사 데이터 전처리 완료")
    list_cols = [f'{cols}_list' for cols in seq_cols]
    int_cols = [
        'A6-1', 'A7-1', 'A8-1', 'A8-2', 'A9-1', 'A9-2', 'A9-3', 'A9-4', 'A9-5'
    ]
    out = pd.concat([df.drop(columns=seq_cols + int_cols, errors="ignore"),
                     feats.drop(columns=list_cols, errors='ignore')], axis=1)
    return out

def preprocess_B(train_B):
    df = train_B.copy()

    print("Step 1: Age 파생...")
    df["Age"] = df["Age"].astype(str).str.extract(r'(\d+)').astype(float)
    feats = pd.DataFrame(index=df.index)

    print("Step 2: Sequence 변환...")
    seq_cols = [
        "B1-1","B1-2","B1-3",
        "B2-1","B2-2","B2-3",
        "B3-1","B3-2",
        "B4-1","B4-2",
        "B5-1","B5-2",
        "B6","B7","B8"
    ]
    for col in seq_cols:
        feats[col + '_list'] = df[col].apply(lambda x: safe_fromstring(x, dtype=float))

    print("Step 3: B1 feature 생성...")
    b1_feats = feats.apply(compute_B1_features, axis=1)
    feats = pd.concat([feats, b1_feats], axis=1)

    print("Step 4: B2 feature 생성...")
    b2_feats = feats.apply(compute_B2_features, axis=1)
    feats = pd.concat([feats, b2_feats], axis=1)

    print("Step 5: B3 feature 생성...")
    b3_feats = feats.apply(compute_B3_features, axis=1)
    feats = pd.concat([feats, b3_feats], axis=1)

    print("Step 6: B4 feature 생성...")
    b4_feats = feats.apply(compute_B4_features, axis=1)
    feats = pd.concat([feats, b4_feats], axis=1)

    print("Step 7: B5 feature 생성...")
    b5_feats = feats.apply(compute_B5_features, axis=1)
    feats = pd.concat([feats, b5_feats], axis=1)

    print("Step 8: B6 feature 생성...")
    b6_feats = feats.apply(compute_B6_features, axis=1)
    feats = pd.concat([feats, b6_feats], axis=1)

    print("Step 9: B7 feature 생성...")
    b7_feats = feats.apply(compute_B7_features, axis=1)
    feats = pd.concat([feats, b7_feats], axis=1)

    print("Step 10: B8 feature 생성...")
    b8_feats = feats.apply(compute_B8_features, axis=1)
    feats = pd.concat([feats, b8_feats], axis=1)

    print("Step 11: B8 feature 생성...")
    feats['B9_aud_hit'] = df['B9-1']
    feats['B9_aud_miss'] = df['B9-2']
    feats['B9_aud_fa'] = df['B9-3']
    feats['B9_aud_cr'] = df['B9-4']
    feats['B9_vis_err'] = df['B9-5']

    print("Step 12: B8 feature 생성...")
    feats['B10_aud_hit'] = df['B10-1']
    feats['B10_aud_miss'] = df['B10-2']
    feats['B10_aud_fa'] = df['B10-3']
    feats['B10_aud_cr'] = df['B10-4']
    feats['B10_vis1_err'] = df['B10-5']
    feats['B10_vis2_correct'] = df['B10-6']

    feats = feats.fillna(0)

    print("B 검사 데이터 전처리 완료")
    list_cols = [f'{cols}_list' for cols in seq_cols]
    int_cols = [
        'B9-1', 'B9-2', 'B9-3', 'B9-4', 'B9-5',
        'B10-1', 'B10-2', 'B10-3', 'B10-4', 'B10-5', 'B10-6'
    ]
    out = pd.concat([df.drop(columns=seq_cols + int_cols, errors="ignore"),
                     feats.drop(columns=list_cols, errors='ignore')], axis=1)
    return out

DROP_COLS = ["Test_id","Test","PrimaryKey","TestDate"]

def align_to_model(X_df, model):
    feat_names = list(getattr(model, "feature_name_", []))

    # ✅ DROP_COLS 무조건 제거
    X = X_df.drop(columns=[c for c in DROP_COLS if c in X_df.columns], errors="ignore").copy()

    if not feat_names:
        # DNN 같은 feature_name_ 없는 모델의 경우
        X = X.select_dtypes(include=[np.number])
        return X.fillna(0.0)

    # LGBM 등 feature_name_ 있는 모델 처리
    for c in feat_names:
        if c not in X.columns:
            X[c] = 0.0

    X = X[feat_names]
    return X.apply(pd.to_numeric, errors="coerce").fillna(0.0)

def main():
    TEST_DIR  = "./data"
    MODEL_DIR = "./model"
    OUT_DIR   = "./output"
    SAMPLE_SUB_PATH = os.path.join(TEST_DIR, "sample_submission.csv")
    OUT_PATH  = os.path.join(OUT_DIR, "submission.csv")

    # ---- 모델 로드 ----
    print("Load models...")
    model_A = load_model(os.path.join(MODEL_DIR, "dnn_model_A.h5"))
    model_B = joblib.load(os.path.join(MODEL_DIR, "lgbm_B.pkl"))
    print(" OK.")

    # ---- 테스트 데이터 로드 ----
    print("Load test data...")
    meta = pd.read_csv(os.path.join(TEST_DIR, "test.csv"))
    Araw = pd.read_csv(os.path.join(TEST_DIR, "./test/A.csv"))
    Braw = pd.read_csv(os.path.join(TEST_DIR, "./test/B.csv"))
    print(f" meta={len(meta)}, Araw={len(Araw)}, Braw={len(Braw)}")

    # ---- 매핑 ----
    A_df = meta.loc[meta["Test"] == "A", ["Test_id", "Test"]].merge(Araw, on="Test_id", how="left")
    B_df = meta.loc[meta["Test"] == "B", ["Test_id", "Test"]].merge(Braw, on="Test_id", how="left")
    print(f" mapped: A={len(A_df)}, B={len(B_df)}")

    # ---- 전처리 → 파생 (학습과 동일) ----
    A_feat = preprocess_A(A_df) if len(A_df) else pd.DataFrame()
    B_feat = preprocess_B(B_df) if len(B_df) else pd.DataFrame()

    XA = align_to_model(A_feat, model_A) if len(A_feat) else pd.DataFrame(columns=getattr(model_A,"feature_name_",[]))
    XB = align_to_model(B_feat, model_B) if len(B_feat) else pd.DataFrame(columns=getattr(model_B,"feature_name_",[]))
    print(f" aligned: XA={XA.shape}, XB={XB.shape}")

    # ---- 정규화 (DNN 입력) ----
    scaler_A = joblib.load(os.path.join(MODEL_DIR, "scaler_A.pkl"))
    XA = scaler_A.transform(XA)

    # ---- 예측 ----
    print("Inference Model...")
    predA = model_A.predict(XA).flatten() if len(XA) else np.array([])
    predB = model_B.predict_proba(XB)[:,1] if len(XB) else np.array([])

    # ---- Test_id와 합치기 ----
    subA = pd.DataFrame({"Test_id": A_df["Test_id"].values, "prob": predA})
    subB = pd.DataFrame({"Test_id": B_df["Test_id"].values, "prob": predB})
    probs = pd.concat([subA, subB], axis=0, ignore_index=True)

    # ---- sample_submission 기반 결과 생성 (Label 컬럼에 0~1 확률 채움) ----
    os.makedirs(OUT_DIR, exist_ok=True)
    sample = pd.read_csv(SAMPLE_SUB_PATH)
    # sample의 Test_id 순서에 맞추어 prob 병합
    out = sample.merge(probs, on="Test_id", how="left")
    out["Label"] = out["prob"].astype(float).fillna(0.0)
    out = out.drop(columns=["prob"])

    out.to_csv(OUT_PATH, index=False)
    print(f"✅ Saved: {OUT_PATH} (rows={len(out)})")

if __name__ == "__main__":
    main()

Load models...


 OK.
Load test data...
 meta=10, Araw=4, Braw=6
 mapped: A=4, B=6
Step 1: Age 파생...
Step 2: Sequence 변환...
Step 3: A1 feature 생성...
Step 4: A2 feature 생성...
Step 5: A3 feature 생성...
Step 6: A4 feature 생성...
Step 7: A5 feature 생성...
Step 8: A6 feature 생성...
Step 9: A7 feature 생성...
Step 10: A8 feature 생성...
Step 9: A7 feature 생성...
A 검사 데이터 전처리 완료
Step 1: Age 파생...
Step 2: Sequence 변환...
Step 3: B1 feature 생성...
Step 4: B2 feature 생성...
Step 5: B3 feature 생성...
Step 6: B4 feature 생성...
Step 7: B5 feature 생성...
Step 8: B6 feature 생성...
Step 9: B7 feature 생성...
Step 10: B8 feature 생성...
Step 11: B8 feature 생성...
Step 12: B8 feature 생성...
B 검사 데이터 전처리 완료
 aligned: XA=(4, 46), XB=(6, 35)
Inference Model...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 200ms/step
✅ Saved: ./output/submission.csv (rows=10)
